# Report


## Introduction / Problem Statement
The goal was to build a movie recommendation system based on the MovieLens data. The task was defined as returning a short list of relevant recommendations for a given input movie. The problem was treated as a content-based recommendation problem in which each movie was represented by metadata and aggregated user information instead of working directly on the three raw tables.

The final solution was built in two stages. First, a baseline model was used to retrieve a candidate pool of similar movies. Second, an improved model was used to rerank the candidate pool and increase variation in the final output. The baseline stage was therefore used internally, while the user only received the final recommendation list from the improved stage.

The most relevant concepts were TF-IDF, cosine similarity, nearest neighbors, and KMeans clustering. Cosine similarity between two vectors $x$ and $y$ is defined as

$$
\cos(x,y) = \frac{x \cdot y}{\|x\|\|y\|}
$$

and was used in the baseline stage to find movies that were close to each other in feature space. In the improved stage, KMeans was used. It minimizes the total squared variation within clusters:

$$
\sum_{i=1}^{k} \sum_{x \in C_i} ||x - \mu_i||^2
$$

This formulation matched the task because the candidate pool had to be divided into groups of similar movies before the final list was selected.


## Data Analysis (EDA)
The dataset consisted of three tables: *movies.csv*, *ratings.csv*, and *tags.csv*. These tables were connected through *movieId*. This made it natural to build a movie-level table where each row represented one movie and where genres, aggregated ratings, and aggregated tags were collected in one place.

The EDA showed that the amount of information varied across movies. Not all movies had both rating signals and user tags. Because the improved model depended on *tag_text* as an important part of candidate clustering, movies without ratings or tags were filtered out. The remaining subset was still large enough for modeling.

The EDA also showed that genres and tags played different roles. Genres gave a broad and stable description of movie content and were therefore suitable as baseline features. Tags were more selective but also semantically richer. For that reason, they were first introduced in the improved stage, where the candidate pool had already been restricted to movies that the baseline model considered relevant.


## Model
Preprocessing first built a movie-level table from the raw data. The tables were loaded, cleaned, and aggregated into columns such as *genres_text*, *tag_text*, *mean_rating*, *rating_count*, *clean_title*, and *year_numeric*. The table was then filtered to movies that had both at least one rating and at least one tag. This filtered table was used as the model input.

The baseline model was built as a sklearn pipeline with two steps: a *ColumnTransformer* step and a nearest-neighbor step. The *ColumnTransformer* converted *genres_text* with TF-IDF and scaled the numeric columns *year_numeric*, *mean_rating*, and *rating_count*. A *NearestNeighbors* model with cosine distance and brute-force search was then fitted. The baseline model was not used as the final output, but only to retrieve a candidate pool.

The improved model was built on top of that candidate pool. It reused the same baseline features and additionally included *tag_text* through TF-IDF. The candidate pool was then clustered with KMeans. The final list was constructed by traversing the clusters in order and selecting strong candidates from several different clusters, which reduced the risk that all recommendations would be too similar to each other.

The main hyperparameters were *candidate_pool_size = 20*, *n_candidate_clusters = 5*, and *n_recommendations = 5*. For tag vectorization, *min_df = 5*, *max_df = 0.8*, and *max_features = 5000* were used. The final model was therefore not a single estimator, but a connected flow consisting of preprocessing, baseline retrieval, and improved candidate clustering.


In [2]:
import pandas as pd

from movie_data import build_movie_table
from recommender import build_recommender
from training_model import RecommendationConfig

config = RecommendationConfig()
movie_table = build_movie_table(use_cache=True, refresh_cache=False)
recommender = build_recommender(config=config, use_cache=True, refresh_cache=False)

example_title = "Toy Story"
baseline_candidates = recommender.recommend(example_title, return_baseline=True)
final_recommendations = recommender.recommend(example_title)

summary = pd.DataFrame({
    "metric": ["movies_in_model", "avg_rating_count", "avg_mean_rating"],
    "value": [
        len(movie_table),
        round(movie_table["rating_count"].mean(), 2),
        round(movie_table["mean_rating"].mean(), 3),
    ],
})

comparison = pd.DataFrame({
    "baseline_title": baseline_candidates["title"].head(5).tolist(),
    "final_title": final_recommendations["title"].head(5).tolist(),
})

summary, comparison


(             metric      value
 0   movies_in_model  50154.000
 1  avg_rating_count    671.700
 2   avg_mean_rating      3.081,
                                       baseline_title  \
 0                                              Shrek   
 1  Lord of the Rings: The Fellowship of the Ring,...   
 2                                     Monsters, Inc.   
 3             Lord of the Rings: The Two Towers, The   
 4                                            Aladdin   
 
                                          final_title  
 0                                              Shrek  
 1  Lord of the Rings: The Fellowship of the Ring,...  
 2                                            Aladdin  
 3                 Star Wars: Episode IV - A New Hope  
 4                                       Pulp Fiction  )

## Results
The code cell above shows both the size of the filtered modeling table and how the baseline and final model behaved for one concrete example. In the current version, the filtered movie-level table contained 50,154 movies. The average number of ratings per movie was about 671.7, and the average mean rating was about 3.081.

For the input movie *Toy Story*, the baseline stage produced a strongly similarity-driven candidate list with titles such as *Shrek*, *The Lord of the Rings: The Fellowship of the Ring*, *Monsters, Inc.*, and *Aladdin*. After the improved stage, the final list changed. Titles such as *Star Wars: Episode IV - A New Hope* and *Pulp Fiction* were included while relevance was still preserved. This result shows that candidate clustering made the final list more varied than a pure top-N list from the baseline model.

The final model was therefore considered better than the baseline model as direct output. The baseline model worked well for retrieval, but the improved model produced a final result that matched the assignment better by providing a clearly improved recommendation method.


## Discussion
The model addressed the task in a reasonable way, but several limitations remain. The recommendations are not personalized at the user level, but are instead based only on a selected input movie. This means that the model recommends movies similar to the chosen title rather than movies optimized for a specific user history.

Filtering to movies with both ratings and tags improved data quality for the improved stage, but it also excluded some movies from the model. In addition, the final output is still influenced by popularity because *rating_count* and *mean_rating* are part of both the feature space and the ranking logic. This creates a risk that well-known movies are favored over narrower choices.

Overall, the selected solution was still appropriate for the assignment. A clear baseline was used in the code, but the final output was based on an improved model that combined baseline retrieval with tag-based KMeans clustering. This provided a clear methodological progression from a simpler model to a more developed final solution.
